# Olist Customers – Data Preprocessing

## Objective

Prepare the Olist customers dataset for the Silver layer.

The preprocessing focuses on data quality validation, standardization,
data type definition and reproducible quality checks.

## Granularity

One row represents one `customer_id` record.

## Key Definitions

- `id_customers` is generated as a sequential integer starting at 1 and serves as the technical unique identifier of each Silver record.
- `customer_id`: identifier of a customer record used to establish the
  relationship with orders.
- `customer_unique_id`: identifier representing the underlying customer
  across potentially multiple customer records.

## Processing Steps

1. Load raw data
2. Perform data audit
3. Validate keys and granularity
4. Check missing values and duplicates
5. Standardize string fields
6. Define explicit data types
7. Validate business rules
8. Run Silver data quality checks
9. Export the Silver dataset

## 1. Setup

Import required Python libraries and define project paths.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RAW_PATH = Path("../datasets/raw")
PROCESSED_PATH = Path("../datasets/processed")

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

In [2]:
bronze_customers = pd.read_csv(RAW_PATH / "olist_customers_dataset.csv")

bronze_customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


## 2. Data Audit

The raw dataset is inspected before any transformation is applied.

The audit covers:

- dataset dimensions
- column names
- data types
- missing values
- duplicate records
- key uniqueness
- customer identifier cardinality
- categorical values
- ZIP code plausibility

In [3]:
#Check shape
bronze_customers.shape

(99441, 5)

In [4]:
#Check columns
bronze_customers.columns.tolist()

['customer_id',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state']

In [5]:
#Check datatypes
bronze_customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 11.0 MB


In [6]:
#Check missing values
bronze_customers.isna().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [7]:
bronze_customers.isna().mean().mul(100).round(2)

customer_id                 0.0
customer_unique_id          0.0
customer_zip_code_prefix    0.0
customer_city               0.0
customer_state              0.0
dtype: float64

In [8]:
#Check duplicates
bronze_customers.duplicated().sum()

np.int64(0)

In [10]:
bronze_customers["customer_id"].nunique()

99441

In [9]:
len(bronze_customers)

99441

In [11]:
#Check primary keys
bronze_customers["customer_id"].is_unique

True

In [12]:
#Check unique ids
bronze_customers["customer_unique_id"].nunique()

96096

In [13]:
customer_counts = (
    bronze_customers
    .groupby("customer_unique_id")
    .size()
    .sort_values(ascending=False))

customer_counts.head(10)

customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
1b6c7548a2a1f9037c1fd3ddfed95f33     7
6469f99c1f9dfae7733b25662e7f1782     7
ca77025e7201e3b30c44b472ff346268     7
47c1a3033b8b77b3ab6e109eb4d5fdf3     6
12f5d6e1cbf93dafd9dcc19095df0b3d     6
63cfc61cee11cbe306bff5857d00bfe4     6
dc813062e0fc23409cd255f7f53c7074     6
de34b16117594161a6a89c50b289d35a     6
dtype: int64

In [14]:
#How many unique customers have multiple customer records?
(customer_counts > 1).sum()

np.int64(2997)

In [15]:
#How many customers have more than one record?
customer_counts.value_counts().sort_index()

1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

In [16]:
repeat_customer_records = customer_counts[customer_counts > 1]

repeat_customer_records.describe()

count    2997.000000
mean        2.116116
std         0.516610
min         2.000000
25%         2.000000
50%         2.000000
75%         2.000000
max        17.000000
dtype: float64

In [17]:
repeat_customer_records.sum()

np.int64(6342)

### Customer Record Cardinality

The dataset contains 96,096 unique `customer_unique_id` values across
99,441 customer records.

2,997 customers have more than one customer record, accounting for
6,342 customer records in total.

These repeated `customer_unique_id` values are not considered duplicate
records because `customer_id` remains unique and represents the
customer-record level of this table.

In [18]:
#Check customer state
bronze_customers["customer_state"].value_counts()

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64

In [19]:
bronze_customers["customer_city"].nunique()

4119

In [20]:
#Check zip-code
bronze_customers["customer_zip_code_prefix"].describe()

count    99441.000000
mean     35137.474583
std      29797.938996
min       1003.000000
25%      11347.000000
50%      24416.000000
75%      58900.000000
max      99990.000000
Name: customer_zip_code_prefix, dtype: float64

In [21]:
bronze_customers["customer_zip_code_prefix"].nunique()

14994

In [22]:
(bronze_customers["customer_zip_code_prefix"] < 0).sum()

np.int64(0)

In [23]:
bronze_customers["customer_zip_code_prefix"].isna().sum()

np.int64(0)

### Audit Findings

The raw customers dataset contains 99,441 records and 5 columns.

No missing values were identified and no fully duplicated rows were found.

`customer_id` is unique across all records and therefore represents the
primary key at the customer-record level.

`customer_unique_id` contains repeated values. This is expected because
multiple customer records can belong to the same underlying customer.
Therefore, `customer_unique_id` must not be treated as the primary key of
this table.

The dataset contains 27 state codes, 4,119 unique cities and 14,994 ZIP
code prefixes.

No structural data quality issue requiring record removal was identified

## 3. Cleaning & Standardization

The raw dataset is preserved and all transformations are applied to a
separate working copy.

Because the audit identified no missing values, duplicate records or
primary-key violations, no customer records are removed during
preprocessing.

The following transformations are applied:

- remove leading and trailing whitespace from string fields
- standardize state codes to uppercase
- explicitly define data types
- preserve the original city naming rather than applying aggressive
  text normalization

In [25]:
#Create copy
silver_customers = bronze_customers.copy()

In [26]:
# Create Silver Unique Key
silver_customers.insert(
    0,
    "id_customers",
    range(1, len(silver_customers) + 1)
)


In [27]:
#Check columns
silver_customers.columns.tolist()

['id_customers',
 'customer_id',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state']

### Silver Unique Key

A technical unique key `id_customers` is created for the Silver table.

The original source identifiers `customer_id` and `customer_unique_id` are retained unchanged.

`id_customers` is generated as a sequential integer starting at 1 and serves as the technical unique identifier of each Silver record.

In [28]:
#Identifying string columns and removing spaces
string_columns = [
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state"]

for col in string_columns:
    silver_customers[col] = silver_customers[col].str.strip()

In [29]:
#Normalize 'state' and check
silver_customers["customer_state"] = (
   silver_customers["customer_state"].str.lower())

In [30]:
silver_customers["customer_state"].unique()

<ArrowStringArray>
['sp', 'sc', 'mg', 'pr', 'rj', 'rs', 'pa', 'go', 'es', 'ba', 'ma', 'ms', 'ce',
 'df', 'rn', 'pe', 'mt', 'am', 'ap', 'al', 'ro', 'pb', 'to', 'pi', 'ac', 'se',
 'rr']
Length: 27, dtype: str

In [31]:
silver_customers["customer_state"].nunique()

27

In [32]:
#Normalize 'city'
silver_customers["customer_city"] = (
    silver_customers["customer_city"].str.strip())

### City Standardization Decision

City values are only stripped of leading and trailing whitespace.

No lower-case, title-case or character replacement is applied because
aggressive normalization could alter geographic names or remove
meaningful linguistic information.

Further geographic standardization, if required, should be handled as a
separate data-quality or geospatial transformation.

In [33]:
#Define datatypes
#IDs
silver_customers["customer_id"] = (
    silver_customers["customer_id"].astype("string"))

silver_customers["customer_unique_id"] = (
    silver_customers["customer_unique_id"].astype("string"))

In [34]:
#City
silver_customers["customer_city"] = (
    silver_customers["customer_city"].astype("string"))

In [35]:
#State
silver_customers["customer_state"] = (
    silver_customers["customer_state"].astype("string"))

In [36]:
#ZIP Prefix
silver_customers["customer_zip_code_prefix"] = (
    silver_customers["customer_zip_code_prefix"]
    .astype("string")
    .str.zfill(5))

In [37]:
#Check datatypes
silver_customers.dtypes

id_customers                 int64
customer_id                 string
customer_unique_id          string
customer_zip_code_prefix    string
customer_city               string
customer_state              string
dtype: object

In [38]:
#Compare raw and cleaned dataset
print("Raw rows:", len(silver_customers))
print("Silver rows:", len(silver_customers))

Raw rows: 99441
Silver rows: 99441


# Olist Customers – Data Preprocessing

## Objective

Prepare the Olist customers dataset for the Silver layer.

The preprocessing focuses on data quality validation, standardization,
data type definition and reproducible quality checks.

## Granularity

One row represents one `customer_id` record.

## Key Definitions

- `id_customers` is generated as a sequential integer starting at 1 and serves as the technical unique identifier of each Silver record.
- `customer_id`: identifier of a customer record used to establish the
  relationship with orders.
- `customer_unique_id`: identifier representing the underlying customer
  across potentially multiple customer records.

## Processing Steps

1. Load raw data
2. Perform data audit
3. Validate keys and granularity
4. Check missing values and duplicates
5. Standardize string fields
6. Define explicit data types
7. Validate business rules
8. Run Silver data quality checks
9. Export the Silver dataset

In [39]:
assert len(silver_customers) == len(silver_customers)

In [40]:
#Check if new missing values appeared
silver_customers.isna().sum()

id_customers                0
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [41]:
# Standardize city names:
# - convert to lowercase
# - replace hyphens with spaces
# - remove apostrophes
# - remove leading/trailing whitespace
# - normalize multiple spaces

silver_customers["customer_city"] = (
    silver_customers["customer_city"]
    .str.lower()
    .str.replace("-", " ", regex=False)
    .str.replace("'", "", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [42]:
# Replace incorrect city value
silver_customers.loc[
    silver_customers["customer_city"] == "quilometro 14 do mutum",
    "customer_city"
] = "mutum"

In [43]:
print(
    silver_customers.loc[
        silver_customers["customer_city"] == "mutum",
        "customer_city"
    ].value_counts()
)

customer_city
mutum    9
Name: count, dtype: int64[pyarrow]


In [44]:
#State quality check
valid_states = {
    'sp', 'sc', 'mg', 'pr', 'rj', 'rs', 'pa', 'go', 'es', 'ba', 'ma', 'ms', 'ce',
    'df', 'rn', 'pe', 'mt', 'am', 'ap', 'al', 'ro', 'pb', 'to', 'pi', 'ac', 'se',
    'rr'
}

invalid_states = set(silver_customers["customer_state"].dropna().unique()) - valid_states

invalid_states

set()

In [45]:
#ZIP Prefix quality check

silver_customers["customer_zip_code_prefix"].str.len().eq(5).all()
silver_customers["customer_zip_code_prefix"].str.isnumeric().all()


np.True_

In [46]:
#Primary-Key quality check
assert silver_customers["customer_id"].notna().all()

assert silver_customers["customer_id"].is_unique

## 4. Silver Data Quality Validation

Before exporting the Silver dataset, a quality gate is applied.

The Silver dataset must satisfy the following requirements:

- record count remains unchanged
- `customer_id` is non-null
- `customer_id` is unique
- `customer_unique_id` is non-null
- `customer_state` is non-null
- no missing values remain
- all state codes are valid
- ZIP code prefixes fall within the defined plausibility range

In [47]:
expected_columns = [
    "id_customers",
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
]

assert silver_customers.columns.tolist() == expected_columns

assert len(silver_customers) == len(bronze_customers)

assert silver_customers["id_customers"].notna().all()
assert silver_customers["id_customers"].is_unique
assert silver_customers["id_customers"].min() == 1
assert silver_customers["id_customers"].max() == len(silver_customers)

assert silver_customers["customer_id"].notna().all()
assert silver_customers["customer_id"].is_unique

assert silver_customers["customer_unique_id"].notna().all()

assert silver_customers["customer_city"].str.match(r"^[a-z\s]+$").all()

assert (silver_customers["customer_city"]== silver_customers["customer_city"].str.lower()).all()

assert silver_customers["customer_state"].notna().all()

assert silver_customers.isna().sum().sum() == 0

assert invalid_states == set()

assert silver_customers["customer_zip_code_prefix"].str.len().eq(5).all()
assert silver_customers["customer_zip_code_prefix"].str.isnumeric().all()

print("Silver data quality checks passed successfully.")

Silver data quality checks passed successfully.


## 5. Silver Dataset Export

After successfully passing all data quality checks, the cleaned customer
dataset is exported to the processed data layer.

The raw dataset remains unchanged.

In [48]:
output_path = PROCESSED_PATH / "silver_customers.csv"

silver_customers.to_csv(
    output_path,
    index=False
)

print(f"Silver dataset exported to: {output_path}")

Silver dataset exported to: ..\datasets\processed\silver_customers.csv


## 6. Customers - Preprocessing Summary

The customers dataset successfully passed the preprocessing quality gate.

### Result

- 99,441 customer records retained
- no records removed
- no missing values identified
- `id_customers` integrated and validated as unique key
- `customer_unique_id` correctly treated as the underlying customer identifier
- `customer_id` validated as the primary key at customer-record level
- state codes validated
- string fields standardized
- data types explicitly defined

The Silver customer dataset is now ready to be used as an input for
downstream analysis and integration with the orders dataset.